# DataFrames

## What's covered

- The Spark API hierarchy — where DataFrames sit between RDDs and (Scala) Datasets
- What a DataFrame really is — schema + partitioned rows + four properties
- Schemas: `StructType` vs DDL string, and why inference is unsafe in production
- Five ways to create a DataFrame
- Exploring a DataFrame — the everyday inspection toolkit
- Column references: string, `col()`, `df[...]`, `expr()` — when each fits
- Common DataFrame operations — select, add/rename/drop, filter, nulls, cast/sort/dedupe
- DataFrame ↔ pandas / RDD interop, and the `toPandas()` trap
- Pandas API on Spark (`pyspark.pandas`) — three DataFrame types, conversions, the index gotcha, limitations


## The Spark API hierarchy

Three structured-data APIs in Spark, in increasing order of abstraction:

- **RDD** — the original. Functional, row-by-row. **No Catalyst optimization, no Tungsten codegen.**
- **DataFrame** — a distributed table with a schema. The everyday API.
- **Dataset[T]** — a typed DataFrame with compile-time type safety. **Scala/Java only.**

In PySpark you only get DataFrames. Internally, `DataFrame` is just an alias for `Dataset[Row]` — same engine, no static type parameter.

The important practical fact: **SQL, DataFrame, and (Scala) Dataset all compile through Catalyst and produce identical execution plans.** RDD bypasses Catalyst — that's why it's slower for structured data.

![Spark API Hierarchy](https://raw.githubusercontent.com/schemabotview/apache-spark/main/img/spark-api-hierarchy.svg)


## What a DataFrame really is

A DataFrame is a **distributed table**: one schema (column names + types) shared across the whole dataset, plus rows partitioned across executor memory.

DataFrame = RDD with metadata, where column names are just human-friendly aliases for row indices, and Catalyst is the translation layer between the two.

| schema (applies to every partition) |
|:---|
| `transaction_id STRING`, `card_id STRING`, `merchant_category STRING`, `amount DECIMAL(18,2)`, `status STRING` |

| partition 1 — executor 1 | partition 2 — executor 2 |
|---|---|
| T0001, C001, Groceries, 1200.00, APPROVED | T0007, C004, Travel, 42000.00, APPROVED |
| T0002, C002, Travel, 18500.00, APPROVED | T0008, C003, Groceries, 1850.00, APPROVED |
| T0003, C001, Food, 450.00, APPROVED | T0009, C005, Shopping, 3200.00, APPROVED |

Each partition holds a **subset of rows with all columns** — partitioning is always horizontal (by row), never vertical (by column).

Properties:

- **Immutable** — every transformation returns a new DataFrame.
- **Lazy** — transformations build a plan; actions execute it.
- **Schema-enforced** — every row matches the same column types.
- **Partitioned** — rows split across partitions for parallel processing.


In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("DataFrames")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

spark.version


## Schemas — `StructType` vs DDL string

Two ways to define a schema. Both produce the exact same result; pick the one that reads better in your code.

- **Programmatic** — `StructType([StructField(...)])`. Verbose, explicit, easy to build from code.
- **DDL string** — `"col_a STRING, col_b DECIMAL(18,2), col_c TIMESTAMP"`. Compact, SQL-flavored, easy to paste.

**Always prefer explicit schemas in production.** Schema inference works but it triggers a full scan of the data to decide types — slow on large datasets, and it can guess wrong (e.g., a sparse `is_flagged` column inferred as `string` when you wanted `boolean`).


In [ ]:
from pyspark.sql.types import (
    StructType, StructField, StringType, DecimalType, BooleanType
)

# Programmatic
txn_schema_struct = StructType([
    StructField("transaction_id",    StringType(),       nullable=False),
    StructField("card_id",           StringType(),       nullable=False),
    StructField("customer_id",       StringType(),       nullable=False),
    StructField("merchant_category", StringType()),
    StructField("amount",            DecimalType(18, 2), nullable=False),
    StructField("status",            StringType()),
    StructField("is_flagged",        BooleanType()),
])

# DDL string — same schema, more compact
txn_schema_ddl = (
    "transaction_id STRING NOT NULL, card_id STRING NOT NULL, customer_id STRING NOT NULL, "
    "merchant_category STRING, amount DECIMAL(18,2) NOT NULL, status STRING, is_flagged BOOLEAN"
)

print(txn_schema_struct.simpleString())


## Creating DataFrames

Five common ways. The first three are workhorses for tests; the last two are everyday.

- `spark.createDataFrame(data, schema)` — Python list + explicit schema (preferred).
- `spark.createDataFrame(data)` — Python list + schema inference.
- `spark.range(n)` — fast integer-sequence DataFrame for testing.
- `spark.createDataFrame(pandas_df)` — round-trip from pandas.
- `spark.read.format(...).load(path)` — from real files (covered in notebook 04).


In [ ]:
from decimal import Decimal

# 1. Python list with explicit schema (preferred)
data = [
    ("T0001", "C001", "CUST001", "Groceries",     Decimal("1200.00"),  "APPROVED", False),
    ("T0002", "C002", "CUST002", "Travel",        Decimal("18500.00"), "APPROVED", False),
    ("T0003", "C001", "CUST001", "Food",          Decimal("450.00"),   "APPROVED", False),
    ("T0004", "C003", "CUST003", "Shopping",      Decimal("6700.00"),  "APPROVED", True),
    ("T0005", "C002", "CUST002", "Fuel",          Decimal("2800.00"),  "DECLINED", False),
    ("T0006", "C001", "CUST001", "Entertainment", Decimal("1100.00"),  "APPROVED", False),
    ("T0007", "C004", "CUST004", "Travel",        Decimal("42000.00"), "APPROVED", True),
    ("T0008", "C003", "CUST003", "Groceries",     Decimal("1850.00"),  "APPROVED", False),
]
df = spark.createDataFrame(data, schema=txn_schema_struct)

# 2. Schema inference — convenient but slow on large data
df_inferred = spark.createDataFrame(
    data,
    ["transaction_id", "card_id", "customer_id", "merchant_category",
     "amount", "status", "is_flagged"],
)

# 3. spark.range — fast integer sequence
df_range = spark.range(0, 5).toDF("idx")

# 4. From a pandas DataFrame
import pandas as pd
pdf = pd.DataFrame({"category": ["Food", "Travel"], "weight": [0.3, 0.7]})
df_from_pandas = spark.createDataFrame(pdf)

print("explicit  :", df.count(), "rows")
print("inferred  :", df_inferred.count(), "rows")
print("range     :", df_range.count(), "rows")
print("pandas    :", df_from_pandas.count(), "rows")


## Exploring a DataFrame

The cheatsheet for *what does this DataFrame contain?*

| Method | Returns |
|---|---|
| `df.count()` | Number of rows (action — triggers a job) |
| `df.columns` | Column names as a Python list |
| `df.dtypes` | List of `(name, type_string)` tuples |
| `df.schema` | The full `StructType` object |
| `df.printSchema()` | Pretty-print the schema |
| `df.describe()` | Count, mean, stddev, min, max for numeric columns |
| `df.summary()` | `describe()` + quartiles |
| `df.show(n, truncate=False)` | Print rows; pass `truncate=False` for full content |


In [ ]:
print("rows x cols :", df.count(), "x", len(df.columns))
print("columns     :", df.columns)
print("dtypes      :", df.dtypes)
print()
df.printSchema()
print()
df.show(3, truncate=False)
print()
df.select("amount").describe().show()


## Column references — the four ways

PySpark accepts four styles for naming a column. They are *mostly* interchangeable but each has a place.

| Form | When it shines | When it breaks |
|---|---|---|
| `"col_name"` (string) | Simple `select`, `groupBy`, `sort` | Cannot apply functions or build expressions |
| `col("col_name")` | Building expressions; passing to `when`, `coalesce`, `F.sum` | None — this is the safest default |
| `df["col_name"]` | Disambiguating columns from two DataFrames in a join | Verbose for normal use |
| `expr("col_name * 0.02")` | Pasting a SQL-style snippet | Easy to typo; the string is parsed at runtime |

Rule of thumb: use a **string** when the API just needs a name, **`col()`** when building an expression, **`df["c"]`** when joining, and **`expr()`** sparingly.


In [ ]:
from pyspark.sql.functions import col, expr

# All four below produce the exact same projected DataFrame.
df.select("merchant_category", "amount").show(3)
df.select(col("merchant_category"), col("amount")).show(3)
df.select(df["merchant_category"], df["amount"]).show(3)
df.selectExpr("merchant_category", "amount").show(3)

# Where col() actually matters — building expressions
df.select(col("amount"), (col("amount") * 0.02).alias("processing_fee")).show(3)


## Common DataFrame operations

The day-to-day toolkit. One cell, five families:

- **Select / project** — `select`, `selectExpr`
- **Add / rename / drop** — `withColumn`, `withColumnRenamed`, `drop`
- **Filter** — `filter` ≡ `where`; combine conditions with `&`, `|`, `~`; **parenthesize each condition** (Python operator precedence will mislead you otherwise)
- **Nulls** — `isNull`, `isNotNull`, `dropna`, `fillna`, `coalesce(...)`
- **Cast / sort / dedupe** — `.cast()`, `orderBy(desc(...))`, `distinct()` vs `dropDuplicates(["col"])`


In [ ]:
from pyspark.sql.functions import col, lit, when, upper, coalesce, desc
from pyspark.sql.types import DoubleType

# --- Select / project -------------------------------------------------------
df.select("transaction_id", "merchant_category", "amount").show(3)
df.selectExpr("transaction_id", "merchant_category", "amount * 0.02 AS fee").show(3)

# --- Add / rename / drop ----------------------------------------------------
df_with = (
    df
    .withColumn("amount_double", col("amount").cast(DoubleType()))                      # add new
    .withColumn("status",        upper(col("status")))                                  # replace existing
    .withColumn("tier",          when(col("amount") > 10000, "HIGH").otherwise("STANDARD"))
    .withColumnRenamed("merchant_category", "category")
    .drop("is_flagged")
)
df_with.show(3, truncate=False)

# --- Filter ------------------------------------------------------------------
# Wrap each condition in (); &, |, ~ are bitwise operators with low precedence in Python.
df.filter((col("status") == "APPROVED") & (col("amount") > 1000)).show(3)
df.where(col("status") != "APPROVED").show(3)
df.filter(col("merchant_category").isin("Travel", "Shopping")).show(3)

# --- Nulls -------------------------------------------------------------------
df_nullable = df.withColumn(
    "merchant_category",
    when(col("transaction_id") == "T0005", lit(None)).otherwise(col("merchant_category")),
)
df_nullable.filter(col("merchant_category").isNull()).show()
df_nullable.fillna({"merchant_category": "UNKNOWN"}).show(3)
df_nullable.select(
    coalesce(col("merchant_category"), lit("UNKNOWN")).alias("category")
).show(3)

# --- Cast / sort / dedupe ---------------------------------------------------
df.select(col("amount").cast(DoubleType()).alias("amount_double")).show(3)
df.orderBy(desc("amount")).show(3)
df.select("merchant_category").distinct().show()                  # all-column distinct
df.dropDuplicates(["card_id"]).show(3)                            # keyed dedupe


## DataFrame ↔ pandas / RDD interop

Three escape hatches you should know.

- **`df.toPandas()`** — pulls every row into a single pandas DataFrame *on the driver*. Convenient for plotting and final-mile analysis. **Dangerous on large data — it OOMs the driver.** Always sample or aggregate first. (`collect()` has the same trap on the Spark side — covered in nb 08.)
- **`spark.createDataFrame(pandas_df)`** — the reverse direction; useful for tiny test fixtures.
- **`df.rdd`** — drop down to an RDD of `Row` objects when you need RDD-only operations like `mapPartitions`. The reverse round-trip is `rdd.toDF(schema)`.


In [ ]:
# To pandas — only on small results
small_pdf = df.filter(col("status") == "APPROVED").limit(3).toPandas()
print(type(small_pdf))
print(small_pdf)
print()

# To RDD — element type is Row
first_row = df.rdd.first()
print(type(first_row))
print("merchant_category :", first_row["merchant_category"])
print("amount            :", first_row.amount)


# Pandas API on Spark

`pyspark.pandas` — formerly the **Koalas** project, merged into PySpark in 3.2 — gives you the pandas API on top of Spark. Most pandas code runs nearly unchanged, but the underlying execution is distributed, lazy, and uses the Spark engine.

Why it exists: pandas does not scale beyond a single machine's memory, and rewriting pandas pipelines into Spark DataFrame API is tedious. With the Pandas API on Spark, the same `psdf.groupby(...).mean()` runs on a cluster.

**Import convention:** `import pyspark.pandas as ps` — the `ps` alias mirrors `pd` for pandas. Many examples online still say `import databricks.koalas as ks` from before the merge; that's the old name.


## Three DataFrame types — keep them straight

| Type | Class | Where data lives | When to use |
|---|---|---|---|
| pandas DataFrame | `pandas.DataFrame` | Driver memory only | Final-mile analysis, plotting, small fixtures |
| Pandas-on-Spark DataFrame | `pyspark.pandas.DataFrame` | Distributed across executors | When you want pandas syntax but Spark scale |
| Spark DataFrame | `pyspark.sql.DataFrame` | Distributed across executors | Native Spark API, best performance, most features |

The first lives entirely on the driver. The other two are distributed — same engine, different surface. Pandas-on-Spark adds a pandas-style index on top of a Spark DataFrame.


In [ ]:
import pyspark.pandas as ps

# Build a pandas-on-Spark DataFrame from the existing Spark DataFrame
psdf = df.pandas_api()
print(f"Class : {type(psdf).__name__}")
print(f"Shape : {psdf.shape}")
print(f"Columns : {list(psdf.columns)}")

psdf.head(3)


## Operations look like pandas

Filtering, projecting, grouping — all the common pandas idioms work. Under the hood they translate to Spark DataFrame operations and execute lazily.


In [ ]:
# pandas-style filtering with boolean indexing
approved = psdf[psdf["status"] == "APPROVED"]
print(f"approved txns: {len(approved)}")

# Column selection + value_counts
top_categories = psdf["merchant_category"].value_counts()
print("\nTransactions by category:")
print(top_categories)

# groupby + agg with named aggregations
by_category = (
    psdf.groupby("merchant_category")
        .agg(avg_amount=("amount", "mean"), n=("transaction_id", "count"))
        .sort_values("n", ascending=False)
)
print("\nAverage amount per category:")
print(by_category)


## Converting between the three types

Conversion is the central skill the exam tests. Each direction has a specific method and a cost.

| From → To | Method | Cost |
|---|---|---|
| Spark DF → Pandas-on-Spark | `sdf.pandas_api()` | Cheap — just a wrapper |
| Spark DF → pandas | `sdf.toPandas()` | **Expensive** — all rows to driver |
| Pandas-on-Spark → Spark DF | `psdf.to_spark()` | Cheap — unwraps the wrapper |
| Pandas-on-Spark → pandas | `psdf.to_pandas()` | **Expensive** — all rows to driver |
| pandas → Pandas-on-Spark | `ps.from_pandas(pdf)` | Spark reads the pandas DF and distributes |
| pandas → Spark DF | `spark.createDataFrame(pdf)` | Spark reads the pandas DF and distributes |

Rule of thumb: the methods with the word *pandas* in them (the lowercase singular `pandas`, not the `pandas_api` wrapper) move data to the driver. Treat them like `collect()` — only use them on small or already-aggregated results.


In [ ]:
# Spark DF  →  Pandas-on-Spark DF
psdf = df.pandas_api()
print("Spark → Pandas-on-Spark :", type(psdf).__module__)

# Pandas-on-Spark DF  →  Spark DF
back_to_spark = psdf.to_spark()
print("Pandas-on-Spark → Spark :", type(back_to_spark).__module__)

# Pandas-on-Spark DF  →  pandas DF  (driver materialization — use only after aggregating)
small = psdf.groupby("merchant_category").size().to_pandas()
print("\nPandas-on-Spark → pandas (after aggregation):")
print(small)
print("Type :", type(small).__module__)

# pandas DF  →  Pandas-on-Spark DF
tiny_pdf = pd.DataFrame({"city": ["Austin", "Dallas"], "score": [85, 72]})
psdf_from_pd = ps.from_pandas(tiny_pdf)
print("\npandas → Pandas-on-Spark :", type(psdf_from_pd).__module__)


## Default indexes — the most-tested gotcha

Pandas DataFrames have a built-in row index. Spark DataFrames do not — rows have no inherent order. Pandas-on-Spark has to *fabricate* an index when one isn't provided, and the strategy it picks has performance and correctness implications.

Three strategies, set via `compute.default_index_type`:

| Index type | How it's computed | Cost | Order guarantee |
|---|---|---|---|
| **`sequence`** | Single executor counts `0, 1, 2, ...` | Triggers a non-distributed step | Sequential — matches pandas behaviour |
| **`distributed-sequence`** (current default) | Computed in parallel using partition offsets | Slight overhead | Sequential — same numbers as pandas |
| **`distributed`** | Monotonically increasing IDs (`monotonically_increasing_id`) | Cheapest — fully parallel | **Not sequential** — gaps allowed |

The trap: `distributed` gives you `0, 1, 8589934592, 8589934593, ...` — pandas code that depends on the exact index values (loc-based lookups, joins on index, `.iloc[5]` semantics on an unsorted axis) will silently break. `sequence` is safe but slow on large data. `distributed-sequence` is the modern default and the right answer for most cases.

Set the index type via options:

```python
ps.set_option("compute.default_index_type", "distributed-sequence")
```


In [ ]:
print("Current default index type :", ps.get_option("compute.default_index_type"))

# Demo: same DF read three different ways shows three different index shapes
for idx_type in ["sequence", "distributed-sequence", "distributed"]:
    ps.set_option("compute.default_index_type", idx_type)
    psdf = df.pandas_api()
    head_idx = list(psdf.head(5).index)
    print(f"  {idx_type:22s} : head indices = {head_idx}")

# Reset to the safe modern default
ps.set_option("compute.default_index_type", "distributed-sequence")


## Limitations and gotchas

The API is *close* to pandas, but not identical. The differences matter on the exam.

| Behaviour | pandas | Pandas API on Spark |
|---|---|---|
| Row order | Stable, follows insertion | **Not guaranteed** unless `sort_index()` or `sort_values()` |
| Lazy vs eager | Eager — every op runs immediately | **Lazy** — like Spark, only actions trigger execution |
| `inplace=True` | Modifies in place | Often ignored — Spark DataFrames are immutable |
| Iteration (`for row in df.iterrows()`) | Cheap | Pulls all rows to driver — avoid |
| `df.apply(func, axis=1)` | Runs Python on every row in one process | Runs distributed but each row crosses the Python-JVM boundary — slow on large data |
| `df.loc[...]` with non-existent label | KeyError | May silently return empty — index isn't fully enforced |
| Some methods missing | — | A handful of pandas methods aren't implemented and raise `PandasNotImplementedError` |

**General performance rule:** if a pandas-on-Spark operation maps to a single Spark DataFrame operation (`filter`, `groupBy`, `join`, `withColumn`), it's fast. If it forces a per-row Python callback or requires materializing the whole frame on the driver, it's slow — at that scale you may as well use the native Spark DataFrame API.


In [ ]:
spark.stop()
